In [3]:
using CMPSExcitations

In [4]:
Hsingle_ll(c, μ) = ∫(∂ψ̂' * ∂ψ̂ - μ * ψ̂' * ψ̂ + c * (ψ̂')^2 * ψ̂^2, (-Inf, +Inf));
Hcoupled(c, μ) = ∫(
    (∂ψ̂₁' * ∂ψ̂₁ - μ * ψ̂₁' * ψ̂₁ + c * (ψ̂₁')^2 * ψ̂₁^2 +
     ∂ψ̂₂' * ∂ψ̂₂ - μ * ψ̂₂' * ψ̂₂ + c * (ψ̂₂')^2 * ψ̂₂^2 +
     2 * c * (ψ̂₁') * (ψ̂₂') * ψ̂₂ * ψ̂₁), (-Inf, +Inf));

In [6]:
# canonical basis
function projection_matrix(D, M)
    dim_in = D^2 + D      # input: E (DxD) + F (D)
    dim_out = 2 * D^2     # output: W1, W2 (both DxD)
    P = zeros(ComplexF64, dim_out, dim_in)

    for j in 1:dim_in
        e = zeros(Float64, dim_in)
        e[j] = 1.0

        E = reshape(view(e, 1:D^2), D, D)
        F = view(e, D^2+1:dim_in)

        W1 = E / sqrt(2) - M * Diagonal(F) / M
        W2 = E / sqrt(2) + M * Diagonal(F) / M

        P[:, j] = vcat(vec(W1), vec(W2))
    end

    return P
end

# canonical basis
function excitation_matrix(Heff, D)
    dim = 2 * D^2
    M = zeros(ComplexF64, dim, dim)

    for j in 1:dim
        e = zeros(ComplexF64, dim)
        e[j] = 1.0
        W1 = reshape(view(e, 1:D^2), D, D)
        W2 = reshape(view(e, D^2+1:dim), D, D)
        W1p, W2p = Heff((Constant(W1), Constant(W2)))
        M[:, j] = vcat(vec(W1p[]), vec(W2p[]))
    end

    return M
end

function excitation_matrix_constrained(Heff, M)
    D = size(M, 1) # R = MDᵣ/M
    H = excitation_matrix(Heff, D)
    P = projection_matrix(D, M)

    P' * H * P, P
end

excitation_matrix_constrained (generic function with 1 method)

In [10]:
Ds = [2, 4, 6, 8, 10, 12, 14];
spectrum = zeros(ComplexF64, length(Ds));
c, μ = 10., 5.
tol = 1e-10
p, nvals = 0, 1;

In [ ]:
Threads.@threads for (D, idx) in enumerate(Ds)
    stateLL = find_groundstate(D, Hsingle_ll(c, μ), InfiniteCMPS, optalg=LBFGS(80; verbosity=1, maxiter=7000, gradtol=tol), gradtol=tol)
    HCLL = Hcoupled(c, μ)
    stateCLL = InfiniteCMPS(stateLL.Q, (stateLL.Rs[1] / sqrt(2), stateLL.Rs[1] / sqrt(2)))

    ρ = real(expval(ψ̂₁' * ψ̂₁ + ψ̂₂' * ψ̂₂, stateCLL)[])

    space = InfiniteCMPSExcitationSpace(p, stateCLL, stateCLL)
    Heff, P = excitation_matrix_constrained(excitation_operator(HCLL, space), eigen(stateCLL.Rs[1][]).vectors)
    vals, vecs = eigsolve(Heff, rand(size(Heff, 2)), nvals, :SR)
    spectrum[idx] = vals[1]
end

┌ Info: UniformCMPS ground state: initialization with e = 675.160985384871
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/infinitecmps/groundstate.jl:115
┌ Info: LBFGS: converged after 150 iterations: f = -2.734747817523, ‖∇f‖ = 3.9900e-11
└ @ OptimKit /home/ashankar/.julia/packages/OptimKit/xpmbV/src/lbfgs.jl:138


Optimizing D=4


┌ Info: UniformCMPS ground state: converged after 151 iterations: e = -2.734747817523, ‖∇e‖ = 3.9900e-11
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/infinitecmps/groundstate.jl:127
┌ Info: UniformCMPS ground state: initialization with e = -2.734747817586
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/infinitecmps/groundstate.jl:115
┌ Info: LBFGS: converged after 337 iterations: f = -2.761265509087, ‖∇f‖ = 4.3606e-11
└ @ OptimKit /home/ashankar/.julia/packages/OptimKit/xpmbV/src/lbfgs.jl:138


D = 4 | InfiniteCMPS{Constant{Matrix{Float64}}, 1}
  0.244135 seconds (632.39 k allocations: 29.157 MiB, 6.17% gc time)
---------------
Optimizing D=8
D = 8 | InfiniteCMPS{Constant{Matrix{Float64}}, 1}
  6.232000 seconds (3.34 M allocations: 309.202 MiB, 1.89% gc time)
---------------
Energy density: -2.761265509086877
 Particle density: 0.8729544307588499
 Order parameter: -0.5115772829097959
Energy density: -2.761265509087147
 Particle density: 0.4364772153794186
 Order parameter: -0.361739765846509


┌ Info: UniformCMPS ground state: converged after 338 iterations: e = -2.761265509087, ‖∇e‖ = 4.3606e-11
└ @ CMPSKit /home/ashankar/Documents/PhDstuff/code/TensorNetworks/CMPSKit.jl/src/infinitecmps/groundstate.jl:127


In [ ]:
plot(Ds, spectrum)